# Assignment 12: Predicting Hotel Booking Cancellations  
## Models: Naïve Bayes, Support Vector Machine (SVM), and Neural Network

**Objectives:**
- Understand how to use classification models (Naïve Bayes, SVM, Neural Networks) to predict hotel cancellations.
- Compare models in terms of accuracy, complexity, and business relevance.
- Interpret and communicate model results from a business perspective.

## Business Scenario

You work as a data analyst for a hospitality group that manages both **Resort** and **City Hotels**. One major challenge in operations is the unpredictability of **booking cancellations**, which affects staffing, inventory, and revenue planning.

You’ve been asked to use historical booking data to predict whether a future booking will be canceled. Your insights will help management plan more effectively.


Your task is to:
1. Build and evaluate three models: Naïve Bayes, SVM, and Neural Network.
2. Compare performance.
3. Recommend which model is best suited for the business needs.

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Assignments/assignment_12_bayes_svm_neural.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


## Dataset Description: Hotel Bookings

This dataset contains booking information for two types of hotels: a **city hotel** and a **resort hotel**. Each record corresponds to a single booking and includes various details about the reservation, customer demographics, booking source, and whether the booking was canceled.

**Source**: [GitHub - TidyTuesday: Hotel Bookings](https://github.com/rfordatascience/tidytuesday/blob/master/data/2020/2020-02-11/readme.md)

### Key Use Cases
- Understand customer booking behavior
- Explore factors related to cancellations
- Segment guests based on booking characteristics
- Compare city vs. resort hotel performance

### Data Dictionary

| Variable | Type | Description |
|----------|------|-------------|
| `hotel` | character | Hotel type: City or Resort |
| `is_canceled` | integer | 1 = Canceled, 0 = Not Canceled |
| `lead_time` | integer | Days between booking and arrival |
| `arrival_date_year` | integer | Year of arrival |
| `arrival_date_month` | character | Month of arrival |
| `stays_in_weekend_nights` | integer | Nights stayed on weekends |
| `stays_in_week_nights` | integer | Nights stayed on weekdays |
| `adults` | integer | Number of adults |
| `children` | integer | Number of children |
| `babies` | integer | Number of babies |
| `meal` | character | Type of meal booked |
| `country` | character | Country code of origin |
| `market_segment` | character | Booking source (e.g., Direct, Online TA) |
| `distribution_channel` | character | Booking channel used |
| `is_repeated_guest` | integer | 1 = Repeated guest, 0 = New guest |
| `previous_cancellations` | integer | Past booking cancellations |
| `previous_bookings_not_canceled` | integer | Past bookings not canceled |
| `reserved_room_type` | character | Initially reserved room type |
| `assigned_room_type` | character | Room type assigned at check-in |
| `booking_changes` | integer | Number of booking modifications |
| `deposit_type` | character | Deposit type (No Deposit, Non-Refund, etc.) |
| `agent` | character | Agent ID who made the booking |
| `company` | character | Company ID (if booking through company) |
| `days_in_waiting_list` | integer | Days on the waiting list |
| `customer_type` | character | Booking type: Contract, Transient, etc. |
| `adr` | float | Average Daily Rate (price per night) |
| `required_car_parking_spaces` | integer | Requested parking spots |
| `total_of_special_requests` | integer | Number of special requests made |
| `reservation_status` | character | Final status (Canceled, No-Show, Check-Out) |
| `reservation_status_date` | date | Date of the last status update |

This dataset is ideal for classification, segmentation, and trend analysis exercises.


## 1. Load and Prepare the Hotel Booking Dataset

**Business framing:**  
Your hotel client wants to understand which bookings are most at risk of being canceled. But before modeling, your job is to prepare the data to ensure clean and reliable input.

### Do the following:
- Import data from the hotels dataset into a dataframe (in GitHub go to the DataSets folder and look for `hotels.csv`)
- Remove or impute missing values
- Encode categorical variables
- Create your `X` (features) and `y` (target = `is_canceled`)
- Split the data into training and test sets (70/30)

### In Your Response:
1. How many total rows and columns are in the dataset?
2. What types of features (categorical, numerical) are included?
3. What steps did you take to clean or prepare the data?


In [8]:
# Add code here 🔧
import pandas as pd
from sklearn.model_selection import train_test_split

# load data
df = pd.read_csv("https://raw.githubusercontent.com/Stan-Pugsley/is_4487_base/refs/heads/main/DataSets/hotels.csv")

# 1) total rows and columns
print(df.shape)

# 2) clean missing values
df["children"] = df["children"].fillna(0)

for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].fillna(df[col].mode()[0])

for col in df.select_dtypes(include=["int64", "float64"]).columns:
    df[col] = df[col].fillna(df[col].median())

# 3) drop columns that leak the answer
df = df.drop(["reservation_status", "reservation_status_date"], axis=1)

# 4) split X and y
y = df["is_canceled"]
X = df.drop("is_canceled", axis=1)

# 5) encode categorical variables
X = pd.get_dummies(X, drop_first=True)

# 6) train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

(9404, 32)


In [12]:
print(X.dtypes)

lead_time                        int64
arrival_date_year                int64
arrival_date_week_number         int64
arrival_date_day_of_month        int64
stays_in_weekend_nights          int64
                                 ...  
deposit_type_Non Refund           bool
deposit_type_Refundable           bool
customer_type_Group               bool
customer_type_Transient           bool
customer_type_Transient-Party     bool
Length: 136, dtype: object


### ✍️ Your Response: 🔧
1. The dataset has 9404 rows and 32 columns

2. It has both categorical features (that have been encoded) like hotel, meal, market segment. It also has numerical features like lead time and price.

3. To clean and prepare this data I encoded the categorical varaibles and also filled in any missing values present while also splitting the data into train and test sets.

## 2. Build a Naïve Bayes Model

**Business framing:**  
Naïve Bayes is a quick, baseline model often used for early testing or simple classification problems.

### Do the following:
- Train a Naïve Bayes classifier on your training data
- Use it to predict on your test data
- Print a classification report and confusion matrix

### In Your Response:
1. How well does the model perform?  And what metric is best used to judge the performance?
2. Where might this model be useful for the hotel (e.g. real-time alerts, operational decisions)?


In [13]:
# Add code here 🔧
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, confusion_matrix

# train model
nb = GaussianNB()
nb.fit(X_train, y_train)

# predictions
y_pred = nb.predict(X_test)

# results
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      0.36      0.53      2121
           1       0.34      1.00      0.51       701

    accuracy                           0.52      2822
   macro avg       0.67      0.68      0.52      2822
weighted avg       0.83      0.52      0.52      2822

[[ 766 1355]
 [   2  699]]


### ✍️ Your Response: 🔧
1. This model did not perform very well overall with an accuracy of only around 52%. The best metric to evaluate this model on is recall as the business wants to avoid any sort of missing cancellations.

2. This could be helpful for flagging high-risk bookings in real-time but may be be the best for accurate decision making in general.

## 3. Build a Support Vector Machine (SVM) Model

**Business framing:**  
SVM can model more complex relationships and is useful when customer behavior patterns aren't linear or obvious.

### Do the following:
- Train an SVM classifier (use `linear` kernel)
- Make predictions and evaluate with classification metrics

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.   

### In Your Response:
1. How well does the model perform?  And what metric is best used to judge the performance?
2. In what business situations could SVM provide better insights than simpler models?


In [14]:
# Add code here 🔧
from sklearn.svm import SVC

# train model (linear kernel)
svm = SVC(kernel="linear")
svm.fit(X_train, y_train)

# predictions
y_pred_svm = svm.predict(X_test)

# results
print(classification_report(y_test, y_pred_svm))
print(confusion_matrix(y_test, y_pred_svm))

              precision    recall  f1-score   support

           0       0.91      0.95      0.93      2121
           1       0.81      0.71      0.76       701

    accuracy                           0.89      2822
   macro avg       0.86      0.83      0.84      2822
weighted avg       0.88      0.89      0.88      2822

[[2005  116]
 [ 204  497]]


### ✍️ Your Response: 🔧
1. The SVM model prefromed pretty well with an accuracy of around 89% which is a huge improvement compared to past models. The best metric to look at would be recall for cancellations since the goal again is to find bookings that will be canceled.

2. SVM would be great for more complex customer behavior and improve overbooking decisions and helping staff plan pricing strategies more effectivley.

## 4. Build a Neural Network Model

**Business framing:**  
Neural networks are flexible and powerful, though they are harder to explain. They may work well when subtle patterns exist in the data.

### Do the following:
- Build a MLPClassifier model using the neural_network package from sklearn
- Choose a simple architecture (e.g., 2 hidden layers)
- Evaluate accuracy and performance

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.  

### In Your Response:
1. How does this model compare to the others?
2. Would the business be comfortable using a “black box” model like this? Why or why not?


In [15]:
# Add code here 🔧
from sklearn.neural_network import MLPClassifier

# model (2 hidden layers)
nn = MLPClassifier(hidden_layer_sizes=(10, 10), max_iter=300, random_state=42)

# train
nn.fit(X_train, y_train)

# predict
y_pred_nn = nn.predict(X_test)

# results
print(classification_report(y_test, y_pred_nn))
print(confusion_matrix(y_test, y_pred_nn))

              precision    recall  f1-score   support

           0       0.96      0.87      0.91      2121
           1       0.70      0.90      0.78       701

    accuracy                           0.88      2822
   macro avg       0.83      0.88      0.85      2822
weighted avg       0.90      0.88      0.88      2822

[[1849  272]
 [  73  628]]


### ✍️ Your Response: 🔧
1. This performed better than the naive bayes by a lot and is much more accurate and balanced. However, compared to the SVM model, the accuracy was only 88% but it still had better recall for cancellations overall.

2. The busines may not be super comfortable with using black box models as it is difficult to make preditions. Despite it having a good performance, managers may like a SVM model more as it is easier to explain and trust.

## 5. Compare All Three Models

### Do the following:
- Print and compare the accuracy of Naïve Bayes, SVM, and Neural Network models
- Summarize which model performed best

### In Your Response:
1. Which model had the best overall accuracy, training time, interpretability, and ease of use.
2. Would you recommend this model for deployment, and why?


In [19]:
# Add code here 🔧
print("Naive Bayes:", accuracy_score(y_test, y_pred))
print("SVM:", accuracy_score(y_test, y_pred_svm))
print("Neural Net:", accuracy_score(y_test, y_pred_nn))

Naive Bayes: 0.5191353649893693
SVM: 0.8866052445074415
Neural Net: 0.8777462792345854


### ✍️ Your Response: 🔧
1. I would say the SVM model did the best overall with the highest accuracy of around 89%. It had a training time that was slower than naive bayes but still very simple and easy. Similarly, naive bayes is the easiest to understand but since SVM is only slighly more complex it is comparable.


2. Yes I would recommend SVM for deployment! As mentoined it has a good balance of performance and interpretability with the highest accuracy compared to the other models. It can predict reliably which makes it a super strong choice for business to make decisions.

## 6. Final Business Recommendation

### In Your Response:
1. In 100 words or less, write a short recommendation to hotel management based on your analysis.

Possible info to include:
- Which model do you recommend implementing?
- What business problem does it help solve?
- Are there any risks or limitations?
- What additional data might improve the results in the future?
2. How does this relate to your customized learning outcome you created in canvas?


### ✍️ Your Response: 🔧
1. As mentioned, I would recommend the SVM model because it has the highest accuracy and performance. It can help the hotel predict booking and cancellations reliably in addition to other decisions like pricing and staffing. One risk is that it is not 100% accurate but it did perform much better than other models. To further improve the results, you could add more customer behavior data to the dataset for better prediction accuracy.

2. This assignment connected to my learning outcomes by allowing me to use data analysis and machine learning to find patterns in customer behavior to inform business decisions. It also let me apply data exploration and evaluate mdoels to find the best approach.

## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [6]:
!jupyter nbconvert --to html "assignment_12_OlivaKaitlyn.ipynb"

[NbConvertApp] WARNING | pattern 'assignment_12_bayes_svm_neural.ipynb' matched no files
This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
    Equivalent to: [--Application.show_config_json=True]
--generate-config
    generate default config file
    Equivalent to: [--JupyterApp.generate_config=True]
-y
    Answer yes to any questions instead of prompting.
    Equivalent to: [--JupyterApp.answer_yes=